# Nimbus Bronze to Silver
Attach `nimbus_iq_silver` as this notebook's default lakehouse. Upload `medallion.py` to Bronze `Files/src/data_generator/`.
Reads all 18 Bronze Delta tables and validates types, keys, foreign keys, timestamps, causal bridges, and investigation totals before writing managed Silver tables.
No local Python setup is required. Existing outputs are protected unless `allow_replace = True` explicitly authorizes a full refresh. Writes are per-table, not a cross-table transaction.


In [ ]:
bronze_lakehouse_name = 'nimbus_iq_bronze'
target_lakehouse_name = 'nimbus_iq_silver'
source_workspace_id = ''
source_schema = 'auto'
allow_replace = False


In [ ]:
import importlib.util
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from pyspark.sql import functions as fn

context = notebookutils.runtime.context
workspace_id = context['currentWorkspaceId']
source_workspace_id = source_workspace_id or workspace_id
target = notebookutils.lakehouse.get(target_lakehouse_name, workspace_id)
if str(context.get('defaultLakehouseId')) != str(target.id) or str(context.get('defaultLakehouseWorkspaceId')) != str(workspace_id):
    raise ValueError(f'Attach {target_lakehouse_name} as default and restart the Spark session.')
bronze = notebookutils.lakehouse.get(bronze_lakehouse_name, source_workspace_id)
endpoint = notebookutils.conf.get('trident.onelake.endpoint').replace('https://', '').rstrip('/')
bronze_root = f'abfss://{source_workspace_id}@{endpoint}/{bronze.id}'
with TemporaryDirectory(prefix='nimbus_medallion_') as directory:
    module_path = Path(directory) / 'medallion.py'
    copied = notebookutils.fs.cp(f'{bronze_root}/Files/src/data_generator/medallion.py', module_path.as_uri())
    if copied is False or not module_path.exists():
        raise FileNotFoundError('Upload medallion.py to Bronze Files/src/data_generator first.')
    spec = importlib.util.spec_from_file_location('nimbus_medallion', module_path)
    medallion = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = medallion
    spec.loader.exec_module(medallion)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
print(f'Source: {bronze_lakehouse_name}; destination: {target_lakehouse_name}')


In [ ]:
bronze_tables = medallion.read_layer(spark, notebookutils, bronze_root, 'bronze', source_schema)
silver_tables = medallion.build_silver(bronze_tables)
primary_incident_id = 'INC-2026-ONT-001'
if silver_tables['incident'].where(fn.col('incident_id') == primary_incident_id).count() != 1:
    raise ValueError('The golden incident is missing; generate and validate Bronze first.')
print('PASS: Silver conformance, relationships, timelines, ranges, and investigation evidence')


In [ ]:
row_counts = medallion.write_layer(spark, silver_tables, allow_replace)
stored = {name: spark.table(name) for name in silver_tables}
medallion.validate_silver(stored)
print(json.dumps({'layer': 'silver', 'status': 'PASS', 'tables': row_counts}, sort_keys=True))
display(medallion.incident_totals(stored).where(fn.col('incident_id') == primary_incident_id))
